In [ ]:
"""
Alpha Trading Research Platform

Notebook:
Sprint07_Scanning_Engine

Purpose:
Run the scanning engine to get one ranked list of trade
opportunities across all four strategies, instead of running each
strategy notebook separately and comparing by eye.

Author:
Alison

Version:
0.9
"""

In [ ]:
# =====================================================
# ALPHA v0.9
# Sprint 7 - Scanning Engine
# =====================================================

from alpha.config import DEFAULT_CONFIG
from alpha.data import get_prices, get_monthly_prices, get_benchmark_prices
from alpha.regime import calculate_regime
from alpha.scanner import scan_latest, top_opportunities

In [ ]:
config = DEFAULT_CONFIG
config

In [ ]:
# =====================================================
# DATA
# =====================================================

print("Downloading market data...")

prices = get_prices(config)
monthly_prices = get_monthly_prices(prices)

print("Downloading benchmark for regime filter...")

benchmark_prices = get_benchmark_prices(config)
monthly_benchmark = benchmark_prices.resample("ME").last()
regime = calculate_regime(monthly_benchmark, config)

current_regime = "bullish" if regime.iloc[-1] else "bearish"
print(f"Current regime: {current_regime}")

In [ ]:
# =====================================================
# THIS WEEK'S SCAN
# =====================================================
# Every ticker at least one strategy currently flags, ranked by how
# many strategies agree, gated by the regime filter above.

scan = scan_latest(monthly_prices, regime, config, daily_prices=prices)

if scan.empty:
    print("No opportunities flagged this month.")
else:
    display(scan)

In [ ]:
# =====================================================
# SHORTLIST
# =====================================================
# Sized to how many trades you're actually planning to manage. You said
# roughly 1/week to start - adjust n as that scales up.

shortlist = top_opportunities(scan, n=3)
display(shortlist)

conflicts = scan[scan["conflicting_signal"]] if not scan.empty else scan
if not conflicts.empty:
    print("\nWorth a second look - strategies disagree on direction:")
    display(conflicts)

In [ ]:
# =====================================================
# NOTES
# =====================================================
# This scan is NOT a backtest position - it's telling you what the
# strategies say as of the latest available month, unshifted. Don't
# feed it straight into build_portfolio()/run_backtest() without
# shifting - that's what the backtest engine's internal shift is for,
# and it's on raw signals from alpha/strategies/, not scanner output.
#
# config.risk_per_trade (1% by default, matching what you told me) is
# defined but not yet used to size anything here - the scanner ranks
# opportunities, it doesn't tell you how many shares to buy. That's
# Sprint 10 (Portfolio Management).
#
# Next is Sprint 8 - Dashboard: turn this table into something you
# actually check each week without opening Jupyter.